# Event Association and GP Mixture outlier identification 
This script computes the maximum candidate sets given BatDetect2 events across N receivers. 

We then compute the TDoA candidate sets using GCC

For a continuous trajectory the TDoA between receiver i and j (τ_ij) should be smooth. We use this observation to identify outliers by fitting a Gaussian-Mixture process to the maximum candidate set for a given receiver pair. Any events that do not belong to the smooth Gaussian process are assumed to be outliers. 

This process is stochastic and may find an incorrect local minima. We leave global optimisation as a future task. The user should manually inspect the plots at the end to validate the reasonableness of the results 

In [ ]:
using WAV
using CSV 
using DataFrames

using Turing
using KernelFunctions
using LinearAlgebra
using Distributions
using Optim
using AbstractGPs

using ReverseDiff
using Plots

using MCMCChains

using StatsFuns: softmax
using Revise
using BayesSoundSource

In [ ]:
function event_windows(events, window)
    window_start = 1 - window÷2
    [[window_start, window_start+window-1] .+ event for event ∈ events]
end 

function getrange(vec, start, stop) 
    len = stop - start + 1
    out = zeros(eltype(vec), len)

    range_vec =  max(1, start) : min(length(vec), stop)
    range_out = range_vec .- (start - 1)

    out[range_out] = @view vec[range_vec]
    out
end 

In [ ]:
function centre_scale(As...) 
    As_ = [A .- mean(A) for A ∈ As]
    ρ = maximum(abs.(vcat(As_...)))
    As_ ./ ρ
end 

function posterior_assignment(dist::MixtureModel, y)
    logw    = log.(probs(dist)) .+ logpdf.(components(dist), y)
    return softmax(logw)
    # return logistic(logw[2] - logw[1])
end

@model function model_gp_mix(x, y, L)
    N = length(x)
    
    # ℓ ~ LogNormal(0.0, 1.0)        # lengthscale > 0
    # σ ~ Exponential(0.5)
    # kernel = with_lengthscale(SqExponentialKernel(), ℓ)
    # f ~ GP(kernel, 1e-5)
    
    # whitened latent GP
    z ~ filldist(Normal(), N)
    f := L * z

    π ~ Uniform(0, 1)

    resp = Vector(undef, N)
    
    for i ∈ 1:N 
        dist = MixtureModel([Normal(f[i], σ), Cauchy()], [π, 1-π])
        y[i] ~ dist 
        resp[i] = posterior_assignment(dist, y[i])[2]
    end 

    return resp
end



In [ ]:
# Set up 
# Assume folder contains 
# 1) Synced WAV files ["receiver_$i.WAV" for i ∈ 1:N] 
#   WAV files should have the same sampling frequency (=freq)
# 2) Detected events using BatDetect2 ["events_$i.WAV" for i ∈ 1:N]
folder = "/home/-/Documents/Field-data/bat_oct25/extracts/20251021_173336"


In [ ]:
freq = 192_000
speed_of_sound = 343 # cm/s

microphone_coords = [
    [0.0, 0.0, 0.0],
    [-8.0, 0.0, 0.89],
    [0.0, 8.0, 1.65],
    [8.0, 0.0, 0.92],
    [0.0, -8.0, 1.64]
]

N = length(microphone_coords)

events = map(1:N) do i
    csv_event = CSV.read(joinpath(folder, "events_$i.csv"), DataFrame)
    return csv_event[!, :start_time]
end 

obs = map(1:5) do i 
    ob, f = wavread(joinpath(folder, "receiver_$i.wav"))
    @assert f == freq
    return ob
end     

# We compute the maximum set of possible events detected on all receivers 
M = maximum_delays(microphone_coords, speed_of_sound; inflate=1.05) 
ambiguous, candidate_ToA_sets = candidate_event_paths(events, M)

ids = ["$(i)_$(j)" for i ∈ 1:N for j ∈ i+1:N]

candidate_TDoA_sets = map(candidate_ToA_sets) do candidate_ToA 
    tdoa_set = Float64[]
    candidate_ToA_sample = floor.(Int, candidate_ToA .* freq)
    windows = event_windows(candidate_ToA_sample, 2^12)
    for i ∈ 1:5 
        i_data = getrange(obs[i], windows[i]...)
        for j ∈ i+1:5 
            j_data = getrange(obs[j], windows[j]...)
            
            offset = candidate_ToA_sample[j] - candidate_ToA_sample[i]

            tdoa = BayesSoundSource.GCC_maximum(j_data, i_data, freq, offset, Inf, x -> PHAT(x) .* bandpass(45_000, 55_000, freq)(x))

            push!(tdoa_set, tdoa) ## τ_ij = τ_i - τ_j
        end 
    end 
    tdoa_set
end 

In [ ]:
using Optim 

# GP parameters should be optimised based on manually selected non-outlier observations
known_good_masks = #.!ambiguous
data_x = xs[3][known_good_masks]
data_y = ys[3][known_good_masks]

jitter = 1e-10
results = optimize([0.0, 0.0], GradientDescent()) do θ
    ℓ = exp(θ[1]) 
    σ² = exp(θ[2])^2 + jitter
    kernel = with_lengthscale(SqExponentialKernel(), ℓ)
    gp = GP(kernel)
    -logpdf(gp(data_x, σ²), data_y)
end 
θ = exp.(results.minimizer)
ℓ = θ[1]
σ² = θ[2]^2 + jitter
kernel = with_lengthscale(SqExponentialKernel(), ℓ)
scatter(data_x, data_y, label="Training data")
plot!(range(extrema(data_x)..., 100), 
    posterior(GP(kernel)(data_x, σ²), data_y), 
    label="GP fit", 
    title="ℓ = $(round(ℓ, sigdigits=3)), 
    σ² = $(round(σ², sigdigits=3))")

In [ ]:

filt = Vector{BitVector}(undef, 10) 
post_resp = Vector(undef, 10)
pred_f = Vector{DataFrame}(undef, 10)
p = Vector{Plots.Plot}(undef, 10)

Threads.@threads for z ∈ 1:10

    kernel = with_lengthscale(SqExponentialKernel(), ℓ)
    K = kernelmatrix(kernel, xs[z]) + σ²*I
    L = cholesky(K).L
    
    local model = model_gp_mix(xs[z], ys[z], L)

    init = @vnt begin
                f := ifelse.(ambiguous, rand(Uniform(-1, 1), length(ys[z])), ys[z])
            end

    local chain = sample(model, 
        NUTS(; adtype=AutoReverseDiff(compile=true)), 
        300, 
        initial_params=(InitFromParams(init)); 
        verbose=false,
        chain_type=MCMCChains.Chains)

    local f = group(chain, Symbol("f"))

    post_resp[z] = mean(returned(model, chain))

    filt[z] = post_resp[z] .< 0.1

    pred_f[z] = DataFrame(summarize(f))
end 


for z ∈ 1:10 

    local df = pred_f[z]
    local m = df.mean

    p[z] = plot(subtitle="τ_$(ids[z])", ylims=(-1.1, 1.1))

    plot!(p[z], xs[z], [m m], fillrange=[m - df.std m + df.std], alpha=0.4, color=2, label=["GP 1σ" ""])
    plot!(p[z], xs[z], [m m], fillrange=[m - 2*df.std m + 2*df.std], alpha=0.2, color=2, label=["GP 2σ" ""])

    if any(.!filt[z])
        scatter!(p[z], xs[z][.!filt[z]], ys[z][.!filt[z]], label="Outliers")#; zcolor=post_resp[z][.!filt[z]])
        vline!(p[z],  xs[z][.!filt[z]], label="")
    end 

    if any(filt[z])
        scatter!(p[z], xs[z][filt[z]], ys[z][filt[z]], label="Inliers", marker=:xcross, color=:black)
    end
end 

p_all = plot(size=(3000, 2000),p..., plot_title="$(basename(folder))")

display(p_all)

In [ ]:
## If the above looks reasionable, then save the filtered ToA-TDoA data

A = reduce(hcat, post_resp)
df_outlier = DataFrame(A, Symbol.("outlier_resp_" .* ids))

A = reduce(vcat, permutedims.(candidate_TDoA_sets))
df_tdoa = DataFrame(A, Symbol.("tdoa_" .* ids))

A = reduce(vcat, permutedims.(candidate_ToA_sets))
df_toa = DataFrame(A, Symbol.("toa_" .* ["$i" for i ∈ 1:5]))

df_data = hcat(df_tdoa, df_toa, df_outlier)

CSV.write("$folder/GCC_data.csv", df_data)
savefig(p_all, "$folder/GCC_data.png")
